# v3 on the held-out GAMUS test split — Kaggle T4

Scores the **v3** checkpoint on `gamus/test`, the split neither v3 nor v4 ever
trained or selected on, with the same protocol v4's `test_gamus_test_*` numbers
used: every tile centre-crop plain + TTA, then a full-tile sliding window + TTA
over the first 400. That makes the two rows of the deck's table comparable.

**Why the v3 tree and not v4's `--test_sources`.** v4 kept every parameter name
but changed HeadB's numerics — the pooled feature is RMS-normalised, the width
softmax runs in fp32, the bin widths are floored (`V4_Kaggle/models/heads.py`).
A v3 checkpoint therefore *loads* into v4 without complaint and then predicts
through different bin centres. Wrong numbers, no error. Only the packed store is
shared — `dwdata/packed.py` is byte-identical in both trees.

### Before you run
1. **Attach** the GAMUS PNG dataset (the one with `test/{images,depth}/`).
2. **Add secrets** `HF_TOKEN` (DINOv3-SAT is gated — the architecture is built
   from the hub even though the checkpoint then overwrites every weight).
3. Set `HF_REPO` / `HF_FILE` below to wherever `best.pt` lives on the Hub.
4. Accelerator: **GPU T4** (one is enough; this is inference). Run through
   *Save Version → Save & Run All (Commit)* — the sliding pass is ~2-3 h and an
   interactive session idles out.

Expected wall clock on one T4: pack ~15 min, plain ~10 min, TTA ~35 min,
sliding+TTA over 400 tiles ~2-3 h.


In [ ]:
# ---- what to score, and where it lives -------------------------------
HF_REPO   = ""                 # e.g. "abhaykale/depthwizard-v3"  <- FILL THIS IN
HF_FILE   = "best.pt"          # the file inside that repo
HF_TYPE   = "model"            # "model" or "dataset"

GAMUS_IN  = ""                 # "" = autodetect under /kaggle/input
SPLIT     = "test"
TILES     = 0                  # 0 = all 2861 centre-crop tiles
SLIDING   = 400                # matches v4's test_gamus_test_sliding_tta

DATA_ROOT = "/kaggle/temp/dwdata"
OUT       = "/kaggle/working/v3_test"
REPO_URL  = "https://github.com/AbhayKale332/DepthWizard.git"

assert HF_REPO, "set HF_REPO to the Hub repo holding the v3 best.pt"


## 1 — code + deps

`torch` is left alone: Kaggle's image already carries the CUDA build.


In [ ]:
import os, subprocess, sys, json, time
from pathlib import Path

def sh(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, **kw)

SRC = Path("/kaggle/working/DepthWizard")
if not SRC.exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(SRC)])
V3 = SRC / "FineTunning" / "v3"
assert (V3 / "eval_test.py").is_file(), f"no eval_test.py under {V3}"

sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(V3 / "requirements.txt")])
os.chdir(V3)
print("cwd:", os.getcwd())


In [ ]:
# HF_TOKEN: the encoder is gated, and the checkpoint may be too.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("no Kaggle secret:", e)
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required (gated DINOv3-SAT)"


## 2 — the v3 checkpoint

2.5 GB. It is pulled once into `/kaggle/temp` (not `/kaggle/working`, which is
capped at 20 GB and is what the notebook output persists).


In [ ]:
from huggingface_hub import hf_hub_download

CKPT = hf_hub_download(HF_REPO, HF_FILE, repo_type=HF_TYPE,
                       token=os.environ["HF_TOKEN"],
                       local_dir="/kaggle/temp/ckpt")
print(CKPT, f"{Path(CKPT).stat().st_size / 1e9:.2f} GB")

# The run that produced it — this is what pins decoder_dim / n_bins /
# encoder_feature_indices. eval_test.py refuses to score if the checkpoint
# does not fit the config, rather than half-loading it under strict=False.
RUN_CONFIG = SRC / "Logs" / "v3" / "config.json"
print(json.dumps({k: json.loads(RUN_CONFIG.read_text())[k] for k in
                  ("encoder_model_id", "decoder_dim", "n_bins", "tile_size",
                   "canonical_gsd_m", "tta_scales")}, indent=1))


## 3 — pack `gamus/test`

Straight off the attached PNG mirror. The store this writes is the same layout
the Modal run read; the check below asserts the geometry matches the store v4
was scored on (2861 tiles, 1024 px, 0.33 m).


In [ ]:
if not GAMUS_IN:
    cands = [p for p in Path("/kaggle/input").rglob(f"{SPLIT}/images") if p.is_dir()]
    assert cands, f"no */{SPLIT}/images under /kaggle/input — attach the GAMUS PNG dataset"
    GAMUS_IN = str(cands[0].parent.parent)
print("gamus source:", GAMUS_IN)

sh([sys.executable, "pack_gamus_png.py", "--src", GAMUS_IN, "--out", DATA_ROOT,
    "--splits", SPLIT, "--depth_scale", "auto", "--prime_workers", "4"])


In [ ]:
idx = json.loads((Path(DATA_ROOT) / "gamus" / SPLIT / "index.json").read_text())
print({k: v for k, v in idx.items() if k != "shards"})

# The store v4's test numbers came from: 2861 tiles @ 1024 px / 0.33 m.  A
# different count here means the mirror's test split is not the same set of
# tiles, and the two rows of the table stop being comparable — say so in the
# deck rather than quietly comparing different problems.
if (idx["n"], idx["tile_px"], idx["gsd_m"]) != (2861, 1024, 0.33):
    print(f"[!] geometry differs from v4's test store (2861, 1024, 0.33) — "
          f"got ({idx['n']}, {idx['tile_px']}, {idx['gsd_m']})")


## 4 — score it

`--amp_dtype fp16` is not a detail: v3 trained under bf16 on an H100, and a T4
is Turing — it has no bf16 units, so leaving the trained value in place gets you
an emulated path that is slow at best. v4's numbers were taken under bf16 on an
H100, so expect fourth-decimal differences from precision alone.

`--batch_size 4 --eval_batch_mult 2` is 8 tiles per forward of a ViT-L at 512 px,
which fits a 16 GB T4 with room. Halve it if you see an OOM.


In [ ]:
cmd = [sys.executable, "eval_test.py",
       "--ckpt", CKPT,
       "--run_config", str(RUN_CONFIG),
       "--data_root", DATA_ROOT,
       "--source", "gamus", "--split", SPLIT,
       "--tiles", str(TILES), "--sliding_tiles", str(SLIDING),
       "--out", OUT,
       "--amp_dtype", "fp16",
       "--batch_size", "4", "--eval_batch_mult", "2",
       "--num_workers", "2", "--prefetch_factor", "4"]
t0 = time.time()
sh(cmd)
print(f"total {(time.time() - t0) / 60:.0f} min")


## 5 — the table for the deck

v4's column is read out of `V4_modal/Output/metrics.json` in the repo, i.e. the
actual numbers the Modal H100 run printed — nothing here is retyped.


In [ ]:
m3 = json.loads((Path(OUT) / "test_metrics.json").read_text())
m4 = json.loads((SRC / "FineTunning" / "V4_modal" / "Output" / "metrics.json").read_text())

def row(d):
    g = d["global"]
    return {"RMSE (m)": g["rmse_m"], "MAE (m)": g["mae_m"], "Pearson r": g["pearson_r"],
            "delta1": g["delta1"], "balanced RMSE (m)": d["balanced_rmse_m"],
            "tall bias (m)": d["tall_gt15m"]["bias_m"],
            "flat bias (m)": d["flat_lt1m"]["bias_m"]}

import pandas as pd
rows, names = [], []
for suffix in ("plain", "tta", "sliding_tta"):
    k = f"test_gamus_test_{suffix}"
    if k in m3:
        rows.append(row(m3[k])); names.append(f"v3 {suffix}")
    if k in m4:
        rows.append(row(m4[k])); names.append(f"v4 {suffix}")
tbl = pd.DataFrame(rows, index=names).round(3)
tbl.to_csv(Path(OUT) / "v3_vs_v4_gamus_test.csv")
(Path(OUT) / "v3_vs_v4_gamus_test.md").write_text(tbl.to_markdown())
print(tbl.to_markdown())
print("\nQuote the sliding_tta row: full-tile sliding window at native GSD on"
      " held-out tiles is what a DSM deliverable actually is.")
tbl


In [ ]:
# Per-stratum, for the slide that explains where the error lives.
k = "test_gamus_test_sliding_tta" if "test_gamus_test_sliding_tta" in m3 else "test_gamus_test_tta"
st = pd.DataFrame({
    "v3 RMSE (m)": {s: d["rmse_m"] for s, d in m3[k]["per_stratum"].items() if d.get("n")},
    "v3 bias (m)": {s: d["bias_m"] for s, d in m3[k]["per_stratum"].items() if d.get("n")},
    "v4 RMSE (m)": {s: d["rmse_m"] for s, d in m4[k]["per_stratum"].items() if d.get("n")},
    "v4 bias (m)": {s: d["bias_m"] for s, d in m4[k]["per_stratum"].items() if d.get("n")},
    "px frac": {s: d["n"] / m3[k]["global"]["n"] for s, d in m3[k]["per_stratum"].items() if d.get("n")},
}).round(3)
st.to_csv(Path(OUT) / "per_stratum.csv")
print(st.to_markdown())
st


## 6 — the test report

v4's `viz/report_html.py`, pointed at v3's numbers. `build()` reads
`<dir>/metrics.json` and already prefers `test_*` keys over val ones, so
`test_metrics.json` plus v3's run config is all it needs. There are no training
figures here, so the report shows only the tables and headline cards. The output is
one self-contained HTML file in `OUT`.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "report_html", SRC / "FineTunning" / "V4_Kaggle" / "viz" / "report_html.py")
report_html = importlib.util.module_from_spec(spec)
spec.loader.exec_module(report_html)

rep_dir = Path(OUT) / "report"
rep_dir.mkdir(parents=True, exist_ok=True)
m = json.loads((Path(OUT) / "test_metrics.json").read_text())
m.setdefault("config", json.loads(RUN_CONFIG.read_text()))
(rep_dir / "metrics.json").write_text(json.dumps(m, indent=2))

rep = report_html.build(rep_dir, out_path=Path(OUT) / "test_report.html",
                        title="DepthWizard v3")
# build() hard-codes "validation report" and a v4 footer; this is v3 on test.
rep.write_text(rep.read_text(encoding="utf-8")
               .replace("validation report", "test report")
               .replace("DepthWizard v4 ·", "DepthWizard v3 ·"), encoding="utf-8")
print(rep)